# PaCMAP-MLX: Rijksmuseum Embedding Explorer

Dimensionality reduction using **PaCMAP-MLX** (Pairwise Controlled Manifold Approximation).

PaCMAP differs from UMAP in how it balances local vs. global structure:
- Uses three types of point pairs: **near** (local), **mid-near**, and **further** (global)
- Optimizes in 3 phases with different pair weightings
- Often better at preserving global structure than UMAP

In [ ]:
import numpy as np
import plotly.graph_objects as go
from lib.embeddings import load_embeddings, load_metadata, build_hover_text

## 1. Load Embeddings

In [ ]:
SAMPLE_SIZE = 20_000  # Set to None for full dataset
SEED = 42

art_ids, object_numbers, embeddings = load_embeddings(sample_size=SAMPLE_SIZE, seed=SEED)
print(f"Loaded {len(art_ids):,} embeddings, shape: {embeddings.shape}")

In [ ]:
# Verify MLX GPU
import mlx.core as mx
print(f"MLX default device: {mx.default_device()}")

## 2. Run PaCMAP-MLX

In [ ]:
from pacmap_mlx import PaCMAP

reducer = PaCMAP(
    n_components=2,
    n_neighbors=10,
    verbose=True,
    random_state=SEED,
)

%time coords = reducer.fit_transform(embeddings)
print(f"Output shape: {coords.shape}")

## 3. Interactive Plotly Scatter

In [ ]:
# Load metadata for hover text
meta = load_metadata(art_ids, object_numbers)
hover_texts = build_hover_text(meta, art_ids)

# Color by primary object type
primary_types = [meta[aid]["types"][0] if meta[aid]["types"] else "unknown" for aid in art_ids]
unique_types = sorted(set(primary_types))
print(f"Object types: {len(unique_types)} unique")

In [ ]:
fig = go.Figure()

for obj_type in unique_types:
    mask = [i for i, t in enumerate(primary_types) if t == obj_type]
    fig.add_trace(go.Scattergl(
        x=coords[mask, 0],
        y=coords[mask, 1],
        mode="markers",
        name=f"{obj_type} ({len(mask):,})",
        text=[hover_texts[i] for i in mask],
        customdata=[object_numbers[i] for i in mask],
        marker=dict(size=3, opacity=0.5),
        hovertemplate="%{text}<extra></extra>",
    ))

n = len(art_ids)
fig.update_layout(
    title=f"Rijksmuseum — PaCMAP-MLX ({n:,} artworks, n_neighbors=10)",
    xaxis=dict(title="PaCMAP 1", showgrid=False, zeroline=False),
    yaxis=dict(title="PaCMAP 2", showgrid=False, zeroline=False, scaleanchor="x"),
    hovermode="closest",
    legend=dict(title="Object Type (click to toggle)", font=dict(size=10)),
    paper_bgcolor="#fafafa",
    plot_bgcolor="#fff",
    height=800,
)

fig.show()

## 4. Parameter Sweep

Vary `n_neighbors` and `MN_ratio` (mid-near pair weight).

In [ ]:
from plotly.subplots import make_subplots

params = [
    {"n_neighbors": 5,  "MN_ratio": 0.5},
    {"n_neighbors": 10, "MN_ratio": 0.5},
    {"n_neighbors": 30, "MN_ratio": 0.5},
    {"n_neighbors": 10, "MN_ratio": 2.0},
]

# Map type strings to integers for colorscale
type_to_int = {t: i for i, t in enumerate(unique_types)}
color_ints = [type_to_int[t] for t in primary_types]

fig_sweep = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"n_neighbors={p['n_neighbors']}, MN_ratio={p['MN_ratio']}" for p in params],
)

for idx, p in enumerate(params):
    row, col = divmod(idx, 2)
    print(f"Running PaCMAP with {p}...")
    r = PaCMAP(n_components=2, random_state=SEED, verbose=False, **p)
    c = r.fit_transform(embeddings)

    fig_sweep.add_trace(
        go.Scattergl(
            x=c[:, 0], y=c[:, 1],
            mode="markers",
            marker=dict(size=2, opacity=0.3, color=color_ints, colorscale="Viridis"),
            text=hover_texts,
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
        ),
        row=row + 1, col=col + 1,
    )

fig_sweep.update_layout(height=1000, title_text="PaCMAP Parameter Sweep")
fig_sweep.update_xaxes(showgrid=False, zeroline=False)
fig_sweep.update_yaxes(showgrid=False, zeroline=False)
fig_sweep.show()

## 5. Save Output

In [ ]:
from lib.embeddings import PROJECT_ROOT

output_dir = PROJECT_ROOT / "output"
output_dir.mkdir(exist_ok=True)

# Save interactive HTML
html_path = output_dir / "pacmap-jupyter.html"
fig.write_html(str(html_path), include_plotlyjs="cdn")
print(f"Saved HTML: {html_path} ({html_path.stat().st_size / 1024:.0f} KB)")

# Save coordinates
npz_path = output_dir / "pacmap-jupyter-coords.npz"
np.savez_compressed(
    str(npz_path),
    coords=coords,
    art_ids=np.array(art_ids),
    object_numbers=np.array(object_numbers),
)
print(f"Saved coords: {npz_path} ({npz_path.stat().st_size / 1024:.0f} KB)")